# Neural Network Crop Recommendation Prototype

This notebook trains a PyTorch multilayer perceptron using N, P, K, pH, temperature, and rainfall.

Soil type and soil moisture are not used by this model because their source units and training signal are not approved. The Test split remains sealed. All reported results use Validation.


## 1 Setup


In [ ]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    ARTIFACT_DIR,
    FEATURES,
    RANDOM_STATE,
    TARGET,
    encode_training_validation_targets,
    evaluate_probabilities,
    load_training_validation_splits,
    plot_evaluation_dashboard,
    prediction_table,
    save_evaluation_outputs,
    training_validation_summary,
)

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
print("Python", platform.python_version())
print("Project root", PROJECT_ROOT)


In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import top_k_accuracy_score

torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(1)
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("PyTorch", torch.__version__)
print("Device", DEVICE)
print("PyTorch CPU threads", torch.get_num_threads())


## 2 Frozen Data Split


In [ ]:
train, validation, sealed_test_summary, split_integrity = load_training_validation_splits()
label_encoder, y_train, y_validation = encode_training_validation_targets(train, validation)

X_train = train[FEATURES].copy()
X_validation = validation[FEATURES].copy()

display(training_validation_summary(train, validation))
print("Features", FEATURES)
print("Crop classes", list(label_encoder.classes_))
print("Sealed Test metadata", sealed_test_summary)


In [ ]:
distribution = pd.concat(
    [
        train.assign(data_split="Train"),
        validation.assign(data_split="Validation"),
    ],
    ignore_index=True,
)
class_counts = (
    distribution.groupby([TARGET, "data_split"])
    .size()
    .rename("row_count")
    .reset_index()
)

plt.figure(figsize=(14, 7))
sns.barplot(data=class_counts, x=TARGET, y="row_count", hue="data_split")
plt.title("Frozen Split Class Distribution")
plt.xlabel("Crop Label")
plt.ylabel("Row Count")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 3 Scale Numeric Features


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation).astype(np.float32)

train_dataset = TensorDataset(
    torch.tensor(X_train_scaled), torch.tensor(y_train, dtype=torch.long)
)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE),
)
X_validation_tensor = torch.tensor(X_validation_scaled, device=DEVICE)
y_validation_tensor = torch.tensor(y_validation, dtype=torch.long, device=DEVICE)
print("Scaler fitted on Train only")


## 4 Train Neural Network


In [ ]:
MODEL_NAME = "neural_network"
MAX_EPOCHS = 250
PATIENCE = 30

class CropMLP(nn.Module):
    def __init__(self, input_size, class_count):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, class_count),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = CropMLP(len(FEATURES), len(label_encoder.classes_)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

best_state = None
best_validation_loss = float("inf")
epochs_without_improvement = 0
history_rows = []
training_start = time.perf_counter()

progress = tqdm(range(1, MAX_EPOCHS + 1), desc="Neural Network epochs", unit="epoch")
for epoch in progress:
    model.train()
    training_loss_sum = 0.0
    training_rows = 0
    for batch_features, batch_targets in train_loader:
        batch_features = batch_features.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch_features)
        loss = criterion(logits, batch_targets)
        loss.backward()
        optimizer.step()
        training_loss_sum += loss.item() * len(batch_features)
        training_rows += len(batch_features)

    model.eval()
    with torch.no_grad():
        validation_logits = model(X_validation_tensor)
        validation_loss = criterion(validation_logits, y_validation_tensor).item()
        stage_probabilities = torch.softmax(validation_logits, dim=1).cpu().numpy()
    validation_top_3 = top_k_accuracy_score(
        y_validation,
        stage_probabilities,
        k=3,
        labels=np.arange(len(label_encoder.classes_)),
    )
    training_loss = training_loss_sum / training_rows
    elapsed = time.perf_counter() - training_start
    history_rows.append(
        {
            "iteration": epoch,
            "training_loss": training_loss,
            "validation_loss": validation_loss,
            "validation_top_3": validation_top_3,
            "elapsed_seconds": elapsed,
        }
    )
    progress.set_postfix(
        top_3=f"{validation_top_3:.3f}", validation_loss=f"{validation_loss:.3f}"
    )

    if validation_loss < best_validation_loss - 1e-4:
        best_validation_loss = validation_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    if epochs_without_improvement >= PATIENCE:
        print("Early stopping at epoch", epoch)
        break

training_seconds = time.perf_counter() - training_start
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    validation_probabilities = torch.softmax(model(X_validation_tensor), dim=1).cpu().numpy()
training_history = pd.DataFrame(history_rows)
print("Neural Network training completed in", round(training_seconds, 3), "seconds")
display(training_history.tail())


## 5 Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(training_history["iteration"], training_history["training_loss"], label="Train")
axes[0].plot(training_history["iteration"], training_history["validation_loss"], label="Validation")
axes[0].set_title("Neural Network Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross Entropy Loss")
axes[0].legend()

axes[1].plot(
    training_history["iteration"],
    training_history["validation_top_3"],
    color="#2f7d32",
)
axes[1].set_ylim(0, 1.02)
axes[1].set_title("Neural Network Validation Top 3")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Top 3 Accuracy")
plt.tight_layout()
plt.show()


## 6 Validation Evaluation


In [ ]:
validation_metrics, validation_per_class, validation_matrix, validation_calibration = (
    evaluate_probabilities(y_validation, validation_probabilities, label_encoder)
)
validation_predictions = prediction_table(
    y_validation, validation_probabilities, label_encoder
)

metrics_table = pd.DataFrame(
    {"metric": list(validation_metrics), "value": list(validation_metrics.values())}
)
display(metrics_table)
display(validation_per_class.sort_values("f1-score").reset_index(drop=True))

plot_evaluation_dashboard(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_matrix,
    validation_calibration,
    y_validation,
    validation_probabilities,
    label_encoder,
    training_history,
)


## 7 Save Validation Artifacts


In [ ]:
output_dir = save_evaluation_outputs(
    MODEL_NAME,
    validation_metrics,
    validation_per_class,
    validation_calibration,
    validation_predictions,
    training_history,
    training_seconds,
)
print("Saved evaluation artifacts to", output_dir)
print("Training time seconds", round(training_seconds, 3))
print("Test split used", False)


In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "features": FEATURES,
        "classes": list(label_encoder.classes_),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "data_contract_version": "prototype-six-feature-v1",
    },
    output_dir / "model.pt",
)
joblib.dump(scaler, output_dir / "scaler.joblib")
joblib.dump(label_encoder, output_dir / "label_encoder.joblib")
print("Saved model", output_dir / "model.pt")
